# 01 — Markov Signal Validation

Load the exported signal dataset and examine whether the Markov signal
has statistically meaningful predictive value for forward returns.

**Prerequisite:** Run `scripts/run_signal_validation.py` first.

In [ ]:
import sys
from pathlib import Path

# Allow src imports from the notebook
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from src.validation import signal_return_correlation, signal_quantile_returns, state_return_summary
from src.reporting import dataset_overview, signal_summary_by_state

pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
DATASET_PATH = ROOT / "data" / "results" / "markov_signal_dataset.parquet"
df = pd.read_parquet(DATASET_PATH)
print(dataset_overview(df))

## Signal Distribution by State

In [ ]:
signal_summary_by_state(df)

## Correlation: Signal vs Forward Returns

In [ ]:
for horizon in [5, 10, 20]:
    col = f"future_{horizon}d_return"
    result = signal_return_correlation(df, return_col=col)
    print(f"\n{horizon}d forward return (n={result['n']:,})")
    print(f"  Pearson  r={result['pearson_r']:.4f}  p={result['pearson_p']:.4f}")
    print(f"  Spearman r={result['spearman_r']:.4f}  p={result['spearman_p']:.4f}")

## Quantile Analysis

Split signal into 5 quantiles (Q0=lowest, Q4=highest).
If the signal has predictive value, Q4 should have the highest mean return.

In [ ]:
for horizon in [5, 10, 20]:
    col = f"future_{horizon}d_return"
    result = signal_quantile_returns(df, return_col=col, n_quantiles=5)
    print(f"\n{horizon}d return by signal quantile:")
    display(result)

## State Regime vs Forward Return

In [ ]:
for horizon in [5, 10, 20]:
    col = f"future_{horizon}d_return"
    print(f"\n{horizon}d return by regime state:")
    display(state_return_summary(df, return_col=col))

## Signal Distribution Histogram

In [ ]:
fig = px.histogram(
    df, x="signal", color="ticker",
    nbins=80, barmode="overlay", opacity=0.6,
    title="Markov Signal Distribution by Ticker",
    labels={"signal": "Signal (P(Bull) - P(Bear))"}
)
fig.show()

## Signal vs 5d Return Scatter (sample)

In [ ]:
sample = df.dropna(subset=["future_5d_return"]).sample(min(5000, len(df)), random_state=42)
fig = px.scatter(
    sample, x="signal", y="future_5d_return",
    color="ticker", opacity=0.4,
    title="Signal vs 5-Day Forward Return",
    trendline="ols"
)
fig.show()